In [0]:
# Databricks Notebook source
# COMMAND ----------
%pip install faker --quiet

# COMMAND ----------
import json
import random
import os
import uuid
from datetime import datetime
from faker import Faker

fake = Faker('pt_BR')

# COMMAND ----------
# 1. Configuração de Parâmetros e Widgets (Ideal para Databricks Jobs)
dbutils.widgets.text("execution_date", "", "Data de Execução (YYYY-MM-DD)")
dbutils.widgets.text("catalog_name", "e-commerce", "Catálogo Unity Catalog")

param_date = dbutils.widgets.get("execution_date")
CATALOG_NAME = dbutils.widgets.get("catalog_name")

# Se não passar parâmetro no Job, assume o dia corrente
if not param_date:
    run_date = datetime.now()
else:
    run_date = datetime.strptime(param_date, "%Y-%m-%d")

date_str = run_date.strftime("%Y-%m-%d")
year_str = run_date.strftime("%Y")
month_str = run_date.strftime("%m")
day_str = run_date.strftime("%d")

print(f"Iniciando simulação para a data de negócio: {date_str}")

# COMMAND ----------
# 2. Definição dos Caminhos no Volume
BASE_PATH = f"/Volumes/{CATALOG_NAME}/raw/raw_landing"

def get_partition_path(entity):
    dir_path = f"{BASE_PATH}/{entity}/year={year_str}/month={month_str}/day={day_str}"
    os.makedirs(dir_path, exist_ok=True)
    return f"{dir_path}/{entity}_{date_str}.json"

# COMMAND ----------
# 3. Lote Diário de Novos Clientes (~50 a 150 novos cadastros por dia)
states = ['SP', 'RJ', 'MG', 'RS', 'PR', 'SC', 'BA', 'PE', 'CE', 'DF']
num_new_customers = random.randint(50, 150)
customers_batch = []

# Obtém o maior customer_id existente ou gera base randômica incremental
for _ in range(num_new_customers):
    customers_batch.append({
        "customer_id": random.randint(100000, 999999),
        "name": fake.name(),
        "email": None if random.random() < 0.03 else fake.email(),
        "phone": None if random.random() < 0.05 else fake.phone_number(),
        "state": random.choice(["XX", None, ""]) if random.random() < 0.02 else random.choice(states),
        "created_at": f"{date_str} {random.randint(0, 23):02d}:{random.randint(0, 59):02d}:{random.randint(0, 59):02d}"
    })

# Injeção de 2% de duplicados intencionais no lote
if len(customers_batch) > 10:
    customers_batch.extend(random.sample(customers_batch, 2))

with open(get_partition_path("customers"), "w", encoding="utf-8") as f:
    for item in customers_batch:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"-> Clientes gerados: {len(customers_batch)}")

# COMMAND ----------
# 4. Lote Diário de Produtos (~0 a 5 produtos novos por dia)
num_new_products = random.randint(0, 5)
categories = ['Eletrônicos', 'Moda', 'Casa e Cozinha', 'Esportes', 'Livros', 'Beleza']
products_batch = []

if num_new_products > 0:
    for _ in range(num_new_products):
        price = -10.0 if random.random() < 0.01 else round(random.uniform(20.0, 2000.0), 2)
        products_batch.append({
            "product_id": random.randint(1000, 9999),
            "product_name": f"Produto {fake.word().capitalize()}",
            "category": None if random.random() < 0.02 else random.choice(categories),
            "price": price,
            "is_active": True
        })

    with open(get_partition_path("products"), "w", encoding="utf-8") as f:
        for item in products_batch:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"-> Produtos novos gerados: {len(products_batch)}")

# COMMAND ----------
# 5. Lote Diário de Pedidos (~1.000 a 2.500 pedidos por dia)
num_orders = random.randint(1000, 2500)
payment_methods = ["pix", "credit_card", "boleto", "voucher"]
order_statuses = ["completed", "completed", "completed", "pending", "canceled"]
orders_batch = []

for _ in range(num_orders):
    items_count = random.choices([1, 2, 3, 4], weights=[65, 25, 8, 2])[0]
    items = []
    
    for _ in range(items_count):
        items.append({
            "product_id": random.randint(1, 500) if random.random() > 0.01 else 99999, # chave órfã
            "quantity": 0 if random.random() < 0.01 else random.randint(1, 4),           # erro de negócio
            "unit_price": round(random.uniform(19.90, 899.90), 2)
        })

    method = random.choice(payment_methods)
    orders_batch.append({
        "order_id": f"ORD-{uuid.uuid4().hex[:8].upper()}",
        "customer_id": random.randint(1, 5000) if random.random() > 0.01 else 88888,
        "order_date": f"{date_str} {random.randint(0, 23):02d}:{random.randint(0, 59):02d}:{random.randint(0, 59):02d}",
        "status": random.choice(["UNKNOWN", None]) if random.random() < 0.01 else random.choice(order_statuses),
        "items": items,
        "payment": {
            "method": method,
            "installments": 1 if method in ["pix", "boleto"] else random.randint(1, 10)
        }
    })

# Inserindo algumas duplicatas de pedidos
orders_batch.extend(random.sample(orders_batch, 5))

with open(get_partition_path("orders"), "w", encoding="utf-8") as f:
    for item in orders_batch:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"-> Pedidos gerados: {len(orders_batch)}")

# COMMAND ----------
# 6. Lote Diário de Eventos / Clickstream (~10.000 a 20.000 eventos por dia)
num_events = random.randint(10000, 20000)
event_types = ["page_view", "view_item", "add_to_cart", "remove_from_cart", "checkout_step"]
platforms = ["web", "app_ios", "app_android"]
events_batch = []

for _ in range(num_events):
    events_batch.append({
        "event_id": f"EVT-{uuid.uuid4().hex[:10].upper()}",
        "session_id": f"SESS-{random.randint(10000, 99999)}",
        "customer_id": random.randint(1, 5000) if random.random() > 0.4 else None,
        "event_type": random.choice(event_types),
        "product_id": random.randint(1, 500) if random.random() > 0.3 else None,
        "platform": random.choice(platforms),
        "event_timestamp": f"{date_str} {random.randint(0, 23):02d}:{random.randint(0, 59):02d}:{random.randint(0, 59):02d}"
    })

with open(get_partition_path("events"), "w", encoding="utf-8") as f:
    for item in events_batch:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"-> Eventos gerados: {len(events_batch)}")
print(f"Carga diária para {date_str} finalizada com sucesso!")